# 이미지 설명 요청하기

## 인터넷에 있는 이미지 사진 설명 요청하기
- 원하는 이미지 링크 사용
- 단, API 호출을 통해 서버에 접근 가능한 이미지 링크를 사용하기

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPEN_API_KEY")  # 환경 변수에서 API 키를 가져오기

client = OpenAI(api_key=api_key)  # 오픈AI 클라이언트의 인스턴스 생성

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 이미지에 대해 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://recipe1.ezmember.co.kr/cache/recipe/2016/06/05/1ba7e34cf0daf694f09a3a9539ebdb161.jpg",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response


ChatCompletion(id='chatcmpl-EVouimnrOoIzE5zPT5tadJ6fV6NYK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='이 이미지는 오므라이스 요리를 보여줍니다. 오므라이스는 달걀로 덮여 있으며, 속에는 볶음밥과 햄, 채소 등이 들어가 있습니다. 오믈렛 위에는 케첩이 뿌려져 있으며, 한쪽 끝이 살짝 열려 있어 안의 내용물이 보입니다. 접시에는 오므라이스 외에도 얇게 썬 배와 브로콜리, 그리고 케첩이 함께 담겨 있습니다. 이 요리는 색깔이 다채롭고 맛있어 보입니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1791252588, model='gpt-4o-2024-08-06', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_70aa11d053', usage=CompletionUsage(completion_tokens=128, prompt_tokens=779, total_tokens=907, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens

## 내가 가진 이미지로 설명 요청하기

### base64 인코딩
- 바이너리 데이터 HTTP Request 전송을 위해 인코딩 진행

In [ ]:
import base64

# Function to encode the image
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")  # 이미지를 base64로 인코딩하고 문자열로 변환
    
image_path = "./data/mart.jpg"

# 이미지를 base64로 인코딩
base64_image = encode_image(image_path)

print(base64_image[0:100])

/9j/4AAUSkZJRgABAQEBLAEsAABBTVBG/+EJxEV4aWYAAE1NACoAAAAIAA0BDwACAAAABgAAAKoBEAACAAAACgAAALABEgADAAAA


### GPT API 호출 및 결과 출력

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 이미지에 대해 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'이 이미지는 상점의 진열대를 보여줍니다. 진열대에는 다양한 컵라면과 봉지 라면이 진열되어 있으며, 주로 매운맛 또는 태국식 라면으로 보입니다. 상단에는 빨간 포장지의 봉지 라면이 쌓여 있고, 오른쪽에는 빨간색과 분홍색 포장의 컵라면들이 보입니다. 아래쪽에는 쌀국수 재료로 보이는 제품들이 진열되어 있습니다. 전체적으로 다양한 종류의 아시아 식품이 구비되어 있는 모습입니다.'

### 여러 이미지 비교 분석 요청

In [ ]:
red_dumpling_base64 = encode_image("data/red_dumpling.jpeg")
dumpling_soup_base64 = encode_image("data/dumpling_soup.jpeg")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "두 음식의 차이점을 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{red_dumpling_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{dumpling_soup_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'첫 번째 음식은 매운 소스에 담긴 만두입니다. 이 음식은 주로 매콤한 맛이 특징이며, 붉은 소스가 매운맛을 강조합니다.\n\n두 번째 음식은 국물에 담긴 만두로, 맑은 국물에 계란 지단 등이 추가된 모습입니다. 이 음식은 국물의 깊은 맛과 담백함이 강조됩니다.\n\n두 음식은 모두 만두를 주재료로 하지만, 소스와 국물의 종류에 따라 맛과 향이 크게 다릅니다. 첫 번째는 매운맛, 두 번째는 담백한 맛이 중심입니다.'

## GPT 비전의 한계 알아보기

In [5]:
oecd_rnd_2021_base64 = encode_image("./data/oecd_rnd_2021_large.png")
oecd_rnd_2022_base64 = encode_image("./data/oecd_rnd_2022_large.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "첫번째는 2021년 데이터이고, 두번째는 2022년 데이터입니다. 이 데이터에 대해 설명해주세요. 어떤 변화가 있었나요? 한국 중심으로 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2021_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2022_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content


'2021년과 2022년 데이터를 비교해보면 다음과 같은 변화가 있습니다:\n\n1. **연구개발비**:\n   - 미국, 중국, 일본의 연구개발비는 2021년과 2022년 모두 증가했습니다.\n   - 한국의 경우, 연구개발비는 89,282백만 US달러에서 2022년에는 91,013백만 US달러로 증가했습니다.\n\n2. **GDP 대비 연구개발비 비중**:\n   - 한국의 GDP 대비 연구개발비 비중은 약간 감소한 것을 볼 수 있습니다. 2021년에는 4.93%였으나 2022년에는 4.91%로 감소했습니다.\n   - 다른 주요 국가들의 비율도 대체로 큰 변동이 없으나, 스위스는 5.44%에서 5.56%로 증가했습니다.\n\n3. **전체적인 경향**:\n   - 주요 국가들의 연구개발 투자 규모는 지속적으로 증가하고 있으며, 특히 한국은 여전히 높은 비중을 유지하고 있습니다.\n   - 한국은 GDP 대비 연구개발비 비중에서 여전히 높은 수준을 유지하며, 기술과 혁신에 대한 투자가 지속되고 있음을 보여줍니다.\n\n이러한 데이터는 각국의 경제 및 과학기술 발전 전략의 방향성을 이해하는 데 도움이 됩니다. 한국은 지속적으로 높은 연구개발 투자 비중을 유지하며 기술 발전에 집중하고 있습니다.'

In [6]:
oecd_rnd_2021_base64 = encode_image("./data/oecd_rnd_2021_medium.png")
oecd_rnd_2022_base64 = encode_image("./data/oecd_rnd_2022.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "첫번째는 2021년 데이터이고, 두번째는 2022년 데이터입니다. 이 데이터에 대해 설명해주세요. 어떤 변화가 있었나요? 한국 중심으로 설명해주세요."},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2021_base64}",
                },
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{oecd_rnd_2022_base64}",
                },
            },
        ],
    }
]

response = client.chat.completions.create(
    model="gpt-4o",  # 응답 생성에 사용할 모델 지정
    messages=messages # 대화 기록을 입력으로 전달
)

response.choices[0].message.content

'이 데이터는 각국의 연구개발비와 GDP 대비 연구개발비 비중을 나타냅니다.\n\n### 2021년에서 2022년 사이의 한국의 변화:\n\n1. **연구개발비**: \n   - 2021년 데이터에서는 약 121,739억 달러였지만, 2022년에는 약 87,225억 달러로 줄어들었습니다.\n   - 이는 연구개발비가 감소한 것을 의미합니다.\n\n2. **GDP 대비 연구개발비 비중**:\n   - 2021년에는 3.13%였으나, 2022년에는 2.91%로 감소하였습니다.\n   - 이는 GDP 대비 연구개발 투자의 비중이 줄어든 것을 의미합니다.\n\n### 전반적인 경향:\n- 미국, 중국, 독일 등 주요 국가들은 연구개발비가 전반적으로 증가하거나 유지되고 있으나, 한국은 두 지표 모두 감소했습니다.\n- 연구개발비 절대 금액뿐만 아니라 GDP 대비 비중도 줄어든 것은 한국의 연구개발 투자에 있어 감소가 이루어졌다는 것을 보여줍니다.\n\n이러한 변화는 경제 상황, 정책 변화, 글로벌 연구개발 흐름 등의 다양한 요인에 영향을 받을 수 있습니다.'

같은 그래프를 해상도만 바꿔 분석했는데 한국의 연구개발비가 늘었는지 줄었는지에 대한 답변이 서로 달랐습니다. 그래프에 글씨와 숫자가 많으면 GPT가 읽는 것에 문제가 생길 수 있었고 중요한 수치는 그래프를 통해 확인하여야 한다는 것을 알았습니다.